In [ ]:
# import duckdb
# import glob
# import os

# con = duckdb.connect()
# con.execute("INSTALL sqlite; LOAD sqlite;")
# con.execute("ATTACH 'medical_data.db' AS sqlite_db (TYPE SQLITE);")

# for csv_file in glob.glob("../synthea/output/csv/*.csv"):
#     table_name = os.path.splitext(os.path.basename(csv_file))[0]
#     print(f"Importing {table_name}...")
#     con.execute(f"CREATE TABLE sqlite_db.{table_name} AS SELECT * FROM read_csv_auto('{csv_file}');")

# print("Done!")

In [1]:
import pandas as pd
import numpy as np
from SQL_query import SQLQuery
SQL = SQLQuery("medical_data.db") 
query = """
WITH LATEST_PREGNANCY AS (
    SELECT
        PATIENT,
        MAX(DATE(SUBSTR(START, 1, 10))) AS PREGNANCY_START,
        DATE(MAX(SUBSTR(START, 1, 10)), '+270 days') AS EST_DELIVERY_DATE
    FROM encounters
    WHERE DESCRIPTION = 'Prenatal initial visit (regime/therapy)'
    GROUP BY PATIENT
),
LATEST_PREG_DIA AS (
    SELECT
        o.PATIENT,
        MAX(DATE(SUBSTR(o.DATE, 1, 10))) AS OBSERVATION_DATE,
        MAX(CASE WHEN o.DESCRIPTION = 'Diastolic Blood Pressure' THEN o.VALUE END) AS Diastolic_Blood_Pressure
    FROM observations as o
    INNER JOIN LATEST_PREGNANCY as lp ON o.PATIENT = lp.PATIENT
    WHERE o.DESCRIPTION = 'Diastolic Blood Pressure' AND DATE(SUBSTR(o.DATE, 1, 10)) <= lp.EST_DELIVERY_DATE
    GROUP BY o.PATIENT
),
LATEST_PREG_SYS AS (
    SELECT
        o.PATIENT,
        MAX(DATE(SUBSTR(o.DATE, 1, 10))) AS OBSERVATION_DATE,
        MAX(CASE WHEN o.DESCRIPTION = 'Systolic Blood Pressure' THEN o.VALUE END) AS Systolic_Blood_Pressure
    FROM observations as o
    INNER JOIN LATEST_PREGNANCY as lp ON o.PATIENT = lp.PATIENT
    WHERE o.DESCRIPTION = 'Systolic Blood Pressure' AND DATE(SUBSTR(o.DATE, 1, 10)) <= lp.EST_DELIVERY_DATE
    GROUP BY o.PATIENT
),
DEATH_CAUSE AS (
    SELECT
        d.PATIENT,
        d.VALUE AS CAUSE_OF_DEATH
    FROM observations as d
    WHERE d.DESCRIPTION = 'Cause of Death [US Standard Certificate of Death]'
)
SELECT 
    t1.Id,
    DATE(t1.BIRTHDATE) AS BIRTHDATE,
    DATE(t1.DEATHDATE) AS DEATHDATE,
    t1.RACE,
    t1.ETHNICITY,
    t1.GENDER,
    t1.COUNTY,
    t1.HEALTHCARE_EXPENSES,
    t1.HEALTHCARE_COVERAGE,
    t1.INCOME,
    t1.HEALTHCARE_EXPENSES - t1.INCOME AS RELATIVE_EXPENSES,
    DATE(t2.PREGNANCY_START) AS PREGNANCY_START,
    DATE(t2.EST_DELIVERY_DATE) AS EST_DELIVERY_DATE,
    DATE(t3.OBSERVATION_DATE) AS BP_OBSERVATION_DATE,
    t3.Diastolic_Blood_Pressure,
    t4.Systolic_Blood_Pressure,
    t8.CAUSE_OF_DEATH
    
FROM patients as t1
INNER JOIN LATEST_PREGNANCY as t2 ON t1.Id = t2.PATIENT
LEFT JOIN LATEST_PREG_DIA as t3 ON t1.Id = t3.PATIENT
LEFT JOIN LATEST_PREG_SYS as t4 ON t1.Id = t4.PATIENT
LEFT JOIN DEATH_CAUSE as t8 ON t1.Id = t8.PATIENT
"""
LAfemMed = SQL.execute_query(query)
LAfemMed['BIRTHDATE'] = pd.to_datetime(LAfemMed['BIRTHDATE'])
LAfemMed['DEATHDATE'] = pd.to_datetime(LAfemMed['DEATHDATE'])
LAfemMed['PREGNANCY_START'] = pd.to_datetime(LAfemMed['PREGNANCY_START'])
LAfemMed['EST_DELIVERY_DATE'] = pd.to_datetime(LAfemMed['EST_DELIVERY_DATE'])
LAfemMed['BP_OBSERVATION_DATE'] = pd.to_datetime(LAfemMed['BP_OBSERVATION_DATE'])
LAfemMed['Days_Pregancy_to_Death'] = (LAfemMed['DEATHDATE'] - LAfemMed['PREGNANCY_START']).dt.days
LAfemMed['Maternal_Mortality'] = np.where(LAfemMed['Days_Pregancy_to_Death'] <= 270+42, 1, 0)

Maternal_Deaths = LAfemMed[LAfemMed['Maternal_Mortality'] == 1]

# Preprocess Data

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
numeric_cols = ['INCOME', 'RELATIVE_EXPENSES', 'Diastolic_Blood_Pressure', 'Systolic_Blood_Pressure']
all_cols = ['INCOME', 'RELATIVE_EXPENSES', 'Diastolic_Blood_Pressure', 'Systolic_Blood_Pressure', 'Maternal_Mortality']
final_data = LAfemMed[all_cols].dropna()
scale = StandardScaler()
X = scale.fit_transform(final_data[numeric_cols])
y = final_data['Maternal_Mortality'].to_numpy().astype(float)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [3]:
from sklearn.naive_bayes import GaussianNB
GNB = GaussianNB()
GNB.fit(X_train, y_train)
GNB_preds = GNB.predict(X_test)

In [4]:
from LogisticRegression import LR_class
LR = LR_class()
LR_fit = LR.LogRes(X_train,y_train)
LR_preds = LR_fit.predict(X_test)

In [5]:
from SVM import SVM_class
svm = SVM_class(X_train, y_train)
svm.best_fit()
svm_preds = svm.predict(X_test)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:1102: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan]
  warnings.warn(


Fitting 5 folds for each of 20 candidates, totalling 100 fits


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:1102: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan]
  warnings.warn(


In [ ]:
import RandForest
rf = RandForest.RF_hyperparams(X_train, y_train)
rf.best_fit()
rf_preds =rf.predict(X_test)

In [ ]:
from NNmod import NN_CVmod, NN_PredMod
in_size = X_train.shape[1]
HidLay_combos = [[2], [6, 3], [10, 5, 2]]
for combo in HidLay_combos:
    nn_mort = NN_CVmod(input_size=in_size, hidden_layers=combo, epochs=1000)
    train_loss, valid_loss = nn_mort.NN_CV(X_train, y_train, k=5)
    print(f"Hidden Layers: {combo} | Average Train Loss: {train_loss:.4f} | Average Valid Loss: {valid_loss:.4f}")
nn_mort = NN_PredMod(input_size=in_size, hidden_layers=[6, 3], epochs=1000)
nn_mort.fit(X_train, y_train)
nn_Spreds = nn_mort.predict(X_test)

In [ ]:
from XGBoost import XGBclass
import xgboost as xgb
xgbcv = XGBclass(X, y)
best_params = xgbcv.xgb_Kfold(k=5)
best_params
xgb_mod = xgb.XGBClassifier(**best_params)
xgb_mod.fit(X_train, y_train)
xgb_preds = xgb_mod.predict(X_test)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, PrecisionRecallDisplay
GNB_PRcurve = PrecisionRecallDisplay.from_predictions(y_test, GNB_preds, name="GaussianNB")
GNB_PRcurve.plot()
LR_PRcurve = PrecisionRecallDisplay.from_predictions(y_test, LR_preds, name="Logistic Regression")
LR_PRcurve.plot()
RF_PRcurve = PrecisionRecallDisplay.from_predictions(y_test, rf_preds, name="Random Forest")
RF_PRcurve.plot()
NN_PRcurve = PrecisionRecallDisplay.from_predictions(y_test, nn_Spreds, name="Neural Network")
NN_PRcurve.plot()
XGB_PRcurve = PrecisionRecallDisplay.from_predictions(y_test, xgb_preds, name="XGBoost")
XGB_PRcurve.plot()